# Traditional Image Classification Pipeline

This notebook implements the traditional computer-vision pipeline for species
classification.

The main pipeline consists of:

1. Loading the shared dataset manifest.
2. Extracting SIFT local descriptors.
3. Constructing a visual vocabulary using MiniBatch K-Means.
4. Encoding each image as a Bag-of-Visual-Words histogram.
5. Training a Linear SVM classifier.
6. Training a Random Forest classifier.
7. Comparing validation and test performance.

This notebook does not use PyTorch. It depends only on NumPy, pandas,
Pillow, OpenCV, scikit-learn, matplotlib and joblib.

In [ ]:
# --------------------------------------------------
# 1. Imports
# --------------------------------------------------

from pathlib import Path
from types import SimpleNamespace
import sys
import json
import time
import random

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

In [ ]:
# --------------------------------------------------
# 2. Locate Project Root
# --------------------------------------------------

CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").is_dir():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").is_dir():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Please run this notebook from the project root "
        "or from the notebooks directory."
    )

SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIR}")

In [ ]:
# --------------------------------------------------
# 3. Import Traditional Pipeline Functions
# --------------------------------------------------

from src.metrics import save_result
from src.traditional.dataset import build_manifest_from_folders, get_class_names
from src.traditional.run_features import run_bovw

from src.traditional.bovw import (
    sift_descriptors,
    sample_train_descriptors,
    build_codebook,
    encode_bovw,
    encode_split,
    build_classifier,
    scores_of,
)

print("Traditional pipeline modules imported successfully.")

In [ ]:
# --------------------------------------------------
# 4. Reproducibility
# --------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed: {SEED}")

In [ ]:
# --------------------------------------------------
# 5. Project Paths
# --------------------------------------------------

DATASET_CANDIDATES = [
    PROJECT_ROOT / "select_train_n_val",
    PROJECT_ROOT.parent / "select_train_n_val",
]
DATASET_ROOT = next((path for path in DATASET_CANDIDATES if path.is_dir()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError("Could not locate select_train_n_val")

TRAIN_DIR = DATASET_ROOT / "select_train_mini"
TEST_DIR = DATASET_ROOT / "select_val_mini"
RESULT_DIR = PROJECT_ROOT / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Training folder: {TRAIN_DIR}")
print(f"Official test folder: {TEST_DIR}")
print(f"Results folder: {RESULT_DIR}")

In [ ]:
# --------------------------------------------------
# 6. Experiment Configuration
# --------------------------------------------------

VOCAB_SIZE = 256
DESCRIPTORS_PER_IMAGE = 100
CODEBOOK_IMAGES = 2000
MAX_CODEBOOK_DESCRIPTORS = 200_000

BOVW_NORMALIZATION = "hellinger"

VALIDATION_PER_CLASS = 10
LIMIT_PER_CLASS = None

RUN_VALIDATION_EXPERIMENTS = False
RUN_FINAL_TEST = False

EXPERIMENT_CONFIG = {
    "seed": SEED,
    "vocabulary_size": VOCAB_SIZE,
    "descriptors_per_image": DESCRIPTORS_PER_IMAGE,
    "codebook_images": CODEBOOK_IMAGES,
    "max_codebook_descriptors": MAX_CODEBOOK_DESCRIPTORS,
    "bovw_normalization": BOVW_NORMALIZATION,
    "validation_per_class": VALIDATION_PER_CLASS,
    "limit_per_class": LIMIT_PER_CLASS,
}

pd.Series(EXPERIMENT_CONFIG, name="value")

## Dataset split

The 50 selected training images per class are split into 40 training and 10 internal validation images. The official iNaturalist validation images remain untouched until the final test.

In [ ]:
manifest = build_manifest_from_folders(
    TRAIN_DIR,
    TEST_DIR,
    validation_per_class=VALIDATION_PER_CLASS,
    seed=SEED,
)
class_names = get_class_names(manifest)
split_counts = manifest.groupby("split").size().rename("images")
per_class_counts = manifest.groupby(["class_id", "split"]).size().unstack(fill_value=0)
display(split_counts)
display(per_class_counts.describe())
assert per_class_counts["train"].eq(40).all()
assert per_class_counts["validation"].eq(10).all()
assert per_class_counts["test"].ge(10).all()
print(f"Dataset checks passed for {len(class_names)} classes.")

## Validation experiments

Vocabulary size and classifier are selected only using internal validation macro-F1. Each experiment rebuilds its vocabulary from training images only.

In [ ]:
def make_args(classifier, k, evaluate_split="validation", **overrides):
    settings = {
        "evaluate_split": evaluate_split,
        "limit_per_class": LIMIT_PER_CLASS,
        "classifier": classifier,
        "k": k,
        "per_image": DESCRIPTORS_PER_IMAGE,
        "codebook_images": CODEBOOK_IMAGES,
        "max_codebook_descriptors": MAX_CODEBOOK_DESCRIPTORS,
        "normalization": BOVW_NORMALIZATION,
        "svm_c": 1.0,
        "rf_estimators": 200,
        "class_weight": None,
        "seed": SEED,
    }
    settings.update(overrides)
    return SimpleNamespace(**settings)


def run_experiment(name, args):
    rows, y_score, train_s, test_s = run_bovw(args, manifest, class_names)
    y_true = rows["class_id"].to_numpy(dtype=np.int64)
    y_pred = np.argmax(y_score, axis=1)
    path, metrics = save_result(
        RESULT_DIR, name, y_true, y_pred, y_score, class_names,
        {"train_s": train_s, "test_s": test_s},
        filepaths=rows["filepath"].tolist(),
    )
    record = {"method": name, "split": args.evaluate_split, **metrics, "result_path": str(path)}
    print(name, "macro_f1:", f"{metrics['macro_f1']:.4f}")
    return record, (rows, y_true, y_pred, y_score)


VALIDATION_EXPERIMENTS = [
    ("bovw_k128_linear_svm", make_args("linear_svm", 128)),
    ("bovw_k256_linear_svm", make_args("linear_svm", 256)),
    ("bovw_k512_linear_svm", make_args("linear_svm", 512)),
    ("bovw_k256_random_forest", make_args("random_forest", 256)),
]

experiment_records = []
experiment_outputs = {}
if RUN_VALIDATION_EXPERIMENTS:
    for name, args in VALIDATION_EXPERIMENTS:
        record, output = run_experiment(name, args)
        experiment_records.append(record)
        experiment_outputs[name] = output
else:
    print("Set RUN_VALIDATION_EXPERIMENTS = True to run this section.")

In [ ]:
if experiment_records:
    results_table = pd.DataFrame(experiment_records).sort_values("macro_f1", ascending=False)
    display(results_table[["method", "top1_accuracy", "top5_accuracy", "balanced_accuracy", "macro_f1", "train_s", "test_s"]])
    ordered = results_table.sort_values("macro_f1")
    axis = ordered.plot.barh(x="method", y="macro_f1", legend=False, figsize=(9, 4))
    axis.set_xlabel("Validation macro-F1")
    axis.set_title("SIFT BoVW validation comparison")
    axis.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Run the validation experiments first.")

## Final held-out test

Copy the single best validation configuration below. Do not change it after viewing the test result.

In [ ]:
FINAL_METHOD_NAME = "bovw_k256_linear_svm_final_test"
FINAL_ARGS = make_args("linear_svm", 256, evaluate_split="test")
final_output = None
if RUN_FINAL_TEST:
    final_record, final_output = run_experiment(FINAL_METHOD_NAME, FINAL_ARGS)
    display(pd.DataFrame([final_record]))
else:
    print("Enable RUN_FINAL_TEST only after selecting the best validation configuration.")

## Final-test error analysis

The log-scaled confusion matrix and confident errors support the report discussion.

In [ ]:
if final_output is not None:
    rows, y_true, y_pred, y_score = final_output
    matrix = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    figure, axis = plt.subplots(figsize=(9, 8))
    image = axis.imshow(np.log1p(matrix), cmap="Blues")
    axis.set_title("Final-test confusion matrix (log count)")
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class")
    figure.colorbar(image, ax=axis)
    plt.show()

    confidence = np.max(y_score, axis=1)
    wrong = np.flatnonzero(y_true != y_pred)
    wrong = wrong[np.argsort(confidence[wrong])[-8:]]
    figure, axes = plt.subplots(2, 4, figsize=(16, 8))
    for axis, index in zip(axes.ravel(), wrong):
        with Image.open(rows.iloc[index]["filepath"]) as sample:
            axis.imshow(sample.convert("RGB"))
        axis.set_title(f"True: {class_names[y_true[index]]}\nPred: {class_names[y_pred[index]]}")
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Run the final test before error analysis.")